# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nimrafarooq2006-coder/ml-internship-work/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I tried three simple models: logistic regression, a depth-3 decision
tree, and a random forest. All three are easy to explain, and the
data is a small starter slice. I compare them with my Week 4 rule on
Precision@50 and Precision@100 (how many of the top-ranked pages are
really declining), because a reviewer can only check a limited number
of pages, and on AUC for the whole ranking. I do not choose by
complexity. I report all of them and choose by what helps at the top
of the list.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

Whole clients are held out. 22 clients (19,166 pages) are used to
train and 10 clients (10,834 pages) to test, with no client in both
(checked: 0 overlap). This matters because pages from the same client
look alike, so a random split would make the models look better than
they are. The label is the same proxy as before (trend_direction is
"down") and it is never used as a feature. 55.9% of the test pages
are declining, so random picking scores about 0.559.

In [5]:
import os, subprocess
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    if not os.path.isdir("ml-internship-work"):
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/nimrafarooq2006-coder/ml-internship-work"], check=True)
    path = "ml-internship-work/" + path

df = pd.read_csv(path)
df["declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["declining"].values
groups = df["client_id"]

train_idx, test_idx = next(GroupShuffleSplit(test_size=0.3, random_state=42).split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y[train_idx], y[test_idx]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# My Week 4 rule, scored on the test pages only
aging = (X_te["days_since_last_update"] >= 90).astype(int)
visible = (X_te["impressions_90d"] >= 100).astype(int)
baseline_score = aging * visible * np.log10(X_te["impressions_90d"].clip(lower=1))

print("Train pages:", len(train_idx), "| Test pages:", len(test_idx))
print("Train clients:", groups.iloc[train_idx].nunique(), "| Test clients:", groups.iloc[test_idx].nunique())
print("Clients in both train and test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print("Declining rate, test:", round(y_te.mean(), 3))
print("BASELINE Precision@50:", round(precision_at_k(baseline_score, y_te, 50), 3))
print("BASELINE Precision@100:", round(precision_at_k(baseline_score, y_te, 100), 3))
print("BASELINE AUC:", round(roc_auc_score(y_te, baseline_score), 3))


Train pages: 19166 | Test pages: 10834
Train clients: 22 | Test clients: 10
Clients in both train and test: 0
Declining rate, test: 0.559
BASELINE Precision@50: 0.46
BASELINE Precision@100: 0.45
BASELINE AUC: 0.525


## 3. Train + compare vs my baseline

All three models beat my Week 4 rule on the same test pages. The rule
scored 0.46 on Precision@50 and 0.525 on AUC, below random picking.
Logistic regression is best at the top of the list (0.66 at 50 and
0.73 at 100). The random forest has the best overall ranking (AUC
0.597) but its top 100 (0.55) is no better than random picking, and
the decision tree's top 50 (0.56) equals random picking. The signal
is weak: every AUC is between 0.57 and 0.60. With only 10 test
clients, differences of a few points may be noise. The most complex
model did not win, so complexity alone is not rewarded.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=8,
                                            min_samples_leaf=50, n_jobs=-1, random_state=42),
}

rows = [{"Method": "My Week 4 rule",
         "Precision@50": precision_at_k(baseline_score, y_te, 50),
         "Precision@100": precision_at_k(baseline_score, y_te, 100),
         "AUC": roc_auc_score(y_te, baseline_score)}]

scores = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    s = m.predict_proba(X_te)[:, 1]
    scores[name] = s
    rows.append({"Method": name,
                 "Precision@50": precision_at_k(s, y_te, 50),
                 "Precision@100": precision_at_k(s, y_te, 100),
                 "AUC": roc_auc_score(y_te, s)})

rows.append({"Method": "Random picking (reference)",
             "Precision@50": y_te.mean(), "Precision@100": y_te.mean(), "AUC": 0.5})

results = pd.DataFrame(rows).round(3)
results


,Method,Precision@50,Precision@100,AUC
0,My Week 4 rule,0.460,0.450,0.525
1,Logistic regression,0.660,0.730,0.576
2,Decision tree (depth 3),0.560,0.600,0.568
3,Random forest,0.600,0.550,0.597
4,Random picking (reference),0.559,0.559,0.500


## 4. Errors and interpretation

What the model relies on (logistic regression, permutation importance
on the test clients): content age matters most (0.043), with younger
pages more likely to be marked declining. Days since the last update
is second (0.019), and longer since an update goes with more decline.
Word count and CTR matter a little. Impressions and average position
barely matter (0.003 and 0.0005), which explains why my Week 4 rule,
which ranked by views, was weak. These are associations in
observational data, not causes. Age and days since update are
related (a page is at least as old as its last update), so their
separate coefficients should be read with care.

What the errors look like: of the model's top 100 pages, 73 are
really declining and 27 are not. The 27 wrong picks tended to be older
pages (median 232 days against 148), not updated for longer (183 days
against 104), and with fewer impressions (84 against 429). Most pages
have a CTR of about zero, so CTR cannot separate the two groups. This
is a small sample of 100 pages from 10 clients, so these patterns are
directional only.

Overall the signal is weak (AUC 0.57 to 0.60). The best use of this
model is to put pages in a rough review order, never to decide on its
own.

In [7]:
from sklearn.inspection import permutation_importance

best = models["Logistic regression"]

coef = pd.Series(best.named_steps["logisticregression"].coef_[0], index=features).round(3)
print("Coefficients (features are scaled, so they can be compared):")
print(coef.sort_values())

imp = permutation_importance(best, X_te, y_te, scoring="roc_auc",
                             n_repeats=5, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({"feature": features, "importance": imp.importances_mean,
                       "spread": imp.importances_std}).sort_values("importance", ascending=False).round(4)
print("\nPermutation importance (drop in AUC when the feature is shuffled):")
print(imp_df.to_string(index=False))

s = scores["Logistic regression"]
order = np.argsort(-s)[:100]
top = X_te.iloc[order].copy()
top["declining"] = y_te[order]
print("\nTop 100 pages: really declining:", int(top["declining"].sum()),
      "| wrong (not declining):", int((top["declining"] == 0).sum()))
print("\nMedian feature values, wrong picks vs right picks:")
print(top.groupby("declining")[features].median().round(2))


Coefficients (features are scaled, so they can be compared):
content_age_days         -0.381
ctr                      -0.198
impressions_90d          -0.045
avg_position             -0.011
word_count                0.062
days_since_last_update    0.242
dtype: float64

Permutation importance (drop in AUC when the feature is shuffled):
               feature  importance  spread
      content_age_days      0.0426  0.0026
days_since_last_update      0.0193  0.0015
            word_count      0.0091  0.0006
                   ctr      0.0038  0.0003
       impressions_90d      0.0026  0.0004
          avg_position      0.0005  0.0002

Top 100 pages: really declining: 73 | wrong (not declining): 27

Median feature values, wrong picks vs right picks:
           content_age_days  days_since_last_update  impressions_90d  \
declining                                                              
0                     232.0                   183.0             84.0   
1                     148.0   

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.